# Outage Regime Analysis - Initial Notebook

This notebook is for exploratory analysis of county-level outage regimes.

The pricing engine currently annualizes historical event counts over the source exposure window. That is a useful v0 baseline, but it can hide important differences:

- stable recurring events across many years;
- storm- or catastrophe-concentrated years;
- persistent low-level source behavior;
- short-trigger prices that may be mathematically valid but commercially weak.

Use this notebook to inspect those patterns before changing production pricing logic.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "price_engine").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

REPO_ROOT

## 1. Parameters

Change these first. The default is Harris County, TX on the 45-minute event catalog with an 8-hour trigger.

In [ ]:
CATALOG_ID = "eagle-i-45min"
TRIGGER_HOURS = 8
PAYOUT_DOLLARS = 1000

# Use None to select from SEARCH_TEXT matches. Harris County, TX is 48201.
SELECTED_FIPS = 48201
SEARCH_TEXT = "Harris"

CATALOG_DIR = REPO_ROOT / "price_engine" / "catalogs" / CATALOG_ID
CATALOG_DIR

## 2. Load catalog artifacts

This reads the generated local catalog, not raw EAGLE-I CSVs.

In [ ]:
events_path = CATALOG_DIR / "data" / "events.parquet"
summary_path = CATALOG_DIR / "data" / "county_summary.parquet"
annual_meta_path = CATALOG_DIR / "data" / "annualization_meta.json"
premiums_path = CATALOG_DIR / "pricing" / "county_premiums.csv"

event_cols = [
    "fips",
    "state",
    "county",
    "start_time",
    "duration_hours",
    "n_snapshots",
    "max_customers",
    "mean_customers",
    "year",
]

events = pd.read_parquet(events_path, columns=event_cols)
events["month"] = events["start_time"].dt.month
events["quarter"] = events["start_time"].dt.quarter

county_summary = pd.read_parquet(summary_path)
premiums = pd.read_csv(premiums_path)
annual_meta = json.loads(annual_meta_path.read_text())

YEARS = annual_meta["years_processed"]
SOURCE_OBS_YEARS = float(annual_meta["source_observation_years"])

print(f"events: {len(events):,}")
print(f"counties: {events['fips'].nunique():,}")
print(f"source observation years: {SOURCE_OBS_YEARS:.3f}")
print(f"source window: {annual_meta['source_window_start']} to {annual_meta['source_window_end']}")

## 3. Select county

Use this to find the FIPS if you do not know it.

In [ ]:
def find_counties(text: str, limit: int = 30) -> pd.DataFrame:
    text = text.strip()
    mask = (
        county_summary["county"].astype(str).str.contains(text, case=False, na=False)
        | county_summary["state"].astype(str).str.contains(text, case=False, na=False)
        | county_summary["fips"].astype(str).str.contains(text, case=False, na=False)
    )
    cols = [
        "fips",
        "state",
        "county",
        "n_events_total",
        "n_per_year",
        "duration_p95",
        "duration_max",
        "mcc",
        "dqi",
    ]
    return county_summary.loc[mask, cols].sort_values("n_events_total", ascending=False).head(limit)

matches = find_counties(SEARCH_TEXT)
display(matches)

if SELECTED_FIPS is None:
    if matches.empty:
        raise ValueError("No county match found. Set SELECTED_FIPS manually.")
    SELECTED_FIPS = int(matches.iloc[0]["fips"])

selected_summary = county_summary[county_summary["fips"] == SELECTED_FIPS].iloc[0]
selected_label = f"{selected_summary['county']} County, {selected_summary['state']} ({SELECTED_FIPS})"
selected_label

## 4. County and trigger profile

This is the core check: does the trigger rate come from many years, or from a small number of concentrated years?

In [ ]:
def county_trigger_profile(events_df: pd.DataFrame, fips: int, trigger_hours: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    county_events = events_df[events_df["fips"] == fips].copy()
    qualifying = county_events[county_events["duration_hours"] >= trigger_hours].copy()

    all_counts = county_events.groupby("year").size().reindex(YEARS, fill_value=0)
    qualifying_counts = qualifying.groupby("year").size().reindex(YEARS, fill_value=0)

    annual = pd.DataFrame(
        {
            "all_events": all_counts,
            "qualifying_events": qualifying_counts,
        }
    )
    annual.index.name = "year"
    annual["qualifying_share_of_all"] = np.where(
        annual["all_events"] > 0,
        annual["qualifying_events"] / annual["all_events"],
        0.0,
    )

    total_q = int(qualifying_counts.sum())
    active_years = int((qualifying_counts > 0).sum())
    sorted_counts = np.sort(qualifying_counts.to_numpy())[::-1]
    top1_share = float(sorted_counts[0] / total_q) if total_q else 0.0
    top2_share = float(sorted_counts[:2].sum() / total_q) if total_q else 0.0
    mean_calendar_year = float(qualifying_counts.mean())
    std_calendar_year = float(qualifying_counts.std(ddof=0))
    cv_calendar_year = std_calendar_year / mean_calendar_year if mean_calendar_year else np.nan

    metrics = {
        "county": selected_label,
        "catalog_id": CATALOG_ID,
        "trigger_hours": trigger_hours,
        "total_events": int(len(county_events)),
        "qualifying_events": total_q,
        "source_observation_years": SOURCE_OBS_YEARS,
        "lambda_T_per_source_year": total_q / SOURCE_OBS_YEARS,
        "active_years_with_qualifying_events": active_years,
        "top_1_year_share": top1_share,
        "top_2_year_share": top2_share,
        "calendar_year_cv": cv_calendar_year,
    }
    return county_events, qualifying, annual, metrics

county_events, qualifying_events, annual_profile, metrics = county_trigger_profile(
    events, SELECTED_FIPS, TRIGGER_HOURS
)

display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
display(annual_profile)

## 5. Visual checks

The first chart shows annual concentration. The second chart shows whether the qualifying events are seasonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

annual_profile["qualifying_events"].plot(kind="bar", ax=axes[0], color="#0f766e")
axes[0].set_title(f"{selected_label}: events >= {TRIGGER_HOURS}h by year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Qualifying events")

if qualifying_events.empty:
    axes[1].text(0.5, 0.5, "No qualifying events", ha="center", va="center")
    axes[1].set_axis_off()
else:
    monthly = (
        qualifying_events.groupby(["year", "month"])
        .size()
        .unstack("month", fill_value=0)
        .reindex(index=YEARS, columns=range(1, 13), fill_value=0)
    )
    image = axes[1].imshow(monthly.to_numpy(), aspect="auto", cmap="YlOrRd")
    axes[1].set_title(f"{selected_label}: qualifying events by month")
    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Year")
    axes[1].set_xticks(range(12), range(1, 13))
    axes[1].set_yticks(range(len(YEARS)), YEARS)
    fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()

## 6. Duration survival and current premium math

This checks whether the short triggers are producing commercially odd prices. The premium table is still the v0 math output; this notebook is for deciding what should be filtered or adjusted later.

In [ ]:
thresholds = [2, 4, 8, 12, 24, 48, 72]
survival_rows = []
for threshold in thresholds:
    n_q = int((county_events["duration_hours"] >= threshold).sum())
    survival_rows.append(
        {
            "T_hours": threshold,
            "n_qualifying": n_q,
            "S_T": n_q / len(county_events) if len(county_events) else 0.0,
            "lambda_T_per_year": n_q / SOURCE_OBS_YEARS,
        }
    )

survival = pd.DataFrame(survival_rows)
display(survival)

premium_view = premiums[
    (premiums["fips"] == SELECTED_FIPS)
    & (premiums["X_dollars"] == PAYOUT_DOLLARS)
    & (premiums["T_hours"].isin(thresholds))
].copy()

display(
    premium_view[
        [
            "T_hours",
            "X_dollars",
            "n_per_year",
            "S_T",
            "lambda_T",
            "pure_premium",
            "retail_premium",
            "tier",
            "quotable",
        ]
    ].sort_values("T_hours")
)

fig, ax = plt.subplots(figsize=(7, 4))
survival.plot(x="T_hours", y="lambda_T_per_year", marker="o", ax=ax, color="#0f766e")
ax.set_title(f"{selected_label}: annual trigger rate by duration threshold")
ax.set_xlabel("Trigger duration T, hours")
ax.set_ylabel("Events per source-year")
ax.grid(True, alpha=0.25)
plt.tight_layout()

## 7. Severity and persistent low-level behavior

This is an early check for whether qualifying events are mostly material outages or persistent small positive observations.

In [ ]:
severity = qualifying_events[
    ["start_time", "duration_hours", "n_snapshots", "max_customers", "mean_customers"]
].copy()

if severity.empty:
    print("No qualifying events for this county and trigger.")
else:
    display(severity.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

    low_severity_share = float((severity["max_customers"] < 100).mean())
    print(f"Share of qualifying events with max_customers < 100: {low_severity_share:.1%}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].hist(severity["duration_hours"], bins=60, color="#0f766e", alpha=0.85)
    axes[0].set_title("Qualifying event duration distribution")
    axes[0].set_xlabel("Duration hours")
    axes[0].set_ylabel("Events")

    axes[1].scatter(
        severity["duration_hours"],
        severity["max_customers"].clip(lower=1),
        s=10,
        alpha=0.35,
        color="#075985",
    )
    axes[1].set_yscale("log")
    axes[1].set_title("Duration vs max customers out")
    axes[1].set_xlabel("Duration hours")
    axes[1].set_ylabel("Max customers out, log scale")
    axes[1].grid(True, alpha=0.25)
    plt.tight_layout()

## 8. First-pass regime flags for all counties

This is intentionally simple. It is a screening layer, not a final underwriting model.

In [ ]:
def classify_regime(row: pd.Series) -> str:
    if row["qualifying_events"] < 25:
        return "thin_history"
    if row["top_1_year_share"] >= 0.50 and row["active_years"] <= 4:
        return "single_year_concentrated"
    if row["top_2_year_share"] >= 0.65:
        return "storm_or_regime_concentrated"
    if row["calendar_year_cv"] >= 1.50:
        return "volatile"
    return "recurring"


def build_regime_summary(events_df: pd.DataFrame, trigger_hours: float) -> pd.DataFrame:
    qevents = events_df[events_df["duration_hours"] >= trigger_hours]
    annual = (
        qevents.groupby(["fips", "year"])
        .size()
        .unstack("year", fill_value=0)
        .reindex(columns=YEARS, fill_value=0)
    )

    total = annual.sum(axis=1)
    active_years = (annual > 0).sum(axis=1)
    sorted_counts = np.sort(annual.to_numpy(), axis=1)[:, ::-1]
    top1_share = np.divide(sorted_counts[:, 0], total.to_numpy(), out=np.zeros(len(total)), where=total.to_numpy() != 0)
    top2_share = np.divide(sorted_counts[:, :2].sum(axis=1), total.to_numpy(), out=np.zeros(len(total)), where=total.to_numpy() != 0)
    mean_year = annual.mean(axis=1)
    std_year = annual.std(axis=1, ddof=0)
    cv = np.divide(std_year, mean_year, out=np.full(len(mean_year), np.nan), where=mean_year.to_numpy() != 0)

    out = pd.DataFrame(
        {
            "fips": annual.index.astype(int).to_numpy(),
            "trigger_hours": trigger_hours,
            "qualifying_events": total.astype(int).to_numpy(),
            "lambda_T_per_source_year": total.to_numpy() / SOURCE_OBS_YEARS,
            "active_years": active_years.astype(int).to_numpy(),
            "top_1_year_share": top1_share,
            "top_2_year_share": top2_share,
            "calendar_year_cv": cv,
        }
    )
    out = out.reset_index(drop=True)
    out = out.merge(county_summary[["fips", "state", "county", "n_events_total", "dqi", "mcc"]], on="fips", how="left")
    out["regime_flag"] = out.apply(classify_regime, axis=1)
    return out.sort_values(["qualifying_events", "top_1_year_share"], ascending=[False, False])


regime_summary = build_regime_summary(events, TRIGGER_HOURS)

display(regime_summary["regime_flag"].value_counts().rename_axis("regime_flag").reset_index(name="counties"))
display(regime_summary.head(25))

## 9. How to use this output

Things to look for:

- If `top_1_year_share` or `top_2_year_share` is high, the simple historical average may be hiding a storm/cat regime.
- If `active_years` is low, the county may need credibility adjustment even if the raw event count looks high.
- If `max_customers` is often small, the trigger may be detecting persistent low-level source behavior rather than economically meaningful outage pain.
- If short-duration premiums are high, pricing may be mathematically valid but commercially weak versus batteries, generators, or resilience upgrades.

This notebook should feed the next pricing design step: routine-rate vs storm-rate decomposition, credibility flags, and commercial viability filters.